In [3]:
#!/usr/bin/env python3
"""Generate a reproducible synthetic archaeological finds database.

Run with: python generate_archaeology_database.py
Optional output path: python generate_archaeology_database.py --output finds.sqlite
"""

from __future__ import annotations

import argparse
import random
import sqlite3
from datetime import date, timedelta
from pathlib import Path


SEED = 20261105
ARTIFACT_COUNT = 1_800
DUPLICATE_SUBMISSIONS = 36
MISSING_SOURCE_NOTES = 144
MISSING_ASSESSMENT_NOTES = 360

MATERIALS = [
    ("CER", "Ceramic"), ("GLS", "Glass"), ("IRON", "Iron"),
    ("COP", "Copper alloy"), ("BONE", "Bone"), ("STONE", "Stone"),
    ("LEAD", "Lead"), ("WOOD", "Wood"), ("LEATH", "Leather"),
    ("OTHER", "Other"),
]

CONSERVATION_SUPPLIES = [
    ("SUP-01", "Deionised water", "cleaning agent"),
    ("SUP-02", "Reversible adhesive", "adhesive"),
    ("SUP-03", "Acid-free tissue", "packing material"),
    ("SUP-04", "Protective wax", "coating"),
    ("SUP-05", "Conservation solvent", "solvent"),
    ("SUP-06", "Acrylic consolidant", "consolidant"),
]

FIND_TYPES = [
    "vessel fragment", "fastener", "tool", "coin", "ornament",
    "building material",
]

CONDITION_WEIGHTS = [0.08, 0.18, 0.34, 0.28, 0.12]

SITES = [
    ("SITE-01", "North Fen", "East Midlands", "rural"),
    ("SITE-02", "Old Market Quarter", "South East", "urban"),
    ("SITE-03", "West Ridge", "South West", "rural"),
    ("SITE-04", "Riverside Works", "North West", "industrial"),
    ("SITE-05", "Hilltop Enclosure", "Yorkshire and Humber", "rural"),
    ("SITE-06", "Harbour Edge", "South West", "coastal"),
]

# Measurement scales used in the generated records:
# - Nominal: material type, find type, site setting, and storage status.
# - Ordinal: condition_grade, ordered 1 Very poor, 2 Poor, 3 Fair,
#   4 Good, 5 Stable.
# - Interval: soil_temperature_c in Celsius; zero is not the absence of heat.
# - Ratio: artefact weight and dimensions, moisture percentage,
#   and treatment quantity.


def connect_database(path: Path) -> sqlite3.Connection:
    """Create a fresh database and enforce its foreign-key relationships."""
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        path.unlink()

    connection = sqlite3.connect(path)
    connection.execute("PRAGMA foreign_keys = ON")
    connection.execute("PRAGMA journal_mode = DELETE")
    return connection


def create_schema(db: sqlite3.Connection) -> None:
    """Create the related tables, keys, and data-quality constraints."""
    db.executescript("""
    CREATE TABLE excavation_sites (
        site_id TEXT PRIMARY KEY,
        site_name TEXT NOT NULL UNIQUE,
        broad_region TEXT NOT NULL,
        site_setting TEXT NOT NULL
            CHECK (site_setting IN ('urban','rural','industrial','coastal'))
    );

    CREATE TABLE trenches (
        trench_id TEXT PRIMARY KEY,
        site_id TEXT NOT NULL REFERENCES excavation_sites(site_id),
        trench_label TEXT NOT NULL,
        area_m2 REAL NOT NULL CHECK (area_m2 > 0),
        UNIQUE (site_id, trench_label)
    );

    CREATE TABLE contexts (
        context_id TEXT PRIMARY KEY,
        trench_id TEXT NOT NULL REFERENCES trenches(trench_id),
        context_number INTEGER NOT NULL CHECK (context_number > 0),
        context_description TEXT NOT NULL,
        deposit_period TEXT NOT NULL
            CHECK (deposit_period IN (
                'prehistoric', 'Roman', 'medieval',
                'post-medieval', 'modern'
            )),
        UNIQUE (trench_id, context_number)
    );

    CREATE TABLE material_types (
        material_id TEXT PRIMARY KEY,
        material_name TEXT NOT NULL UNIQUE
    );

    CREATE TABLE conservation_supplies (
        supply_id TEXT PRIMARY KEY,
        supply_name TEXT NOT NULL UNIQUE,
        supply_type TEXT NOT NULL
            CHECK (supply_type IN (
                'cleaning agent', 'adhesive', 'packing material',
                'coating', 'solvent', 'consolidant'
            ))
    );

    CREATE TABLE find_types (
        find_type_id INTEGER PRIMARY KEY,
        find_type_name TEXT NOT NULL UNIQUE
    );

    CREATE TABLE conservators (
        conservator_id TEXT PRIMARY KEY,
        role_level TEXT NOT NULL
            CHECK (role_level IN (
                'trainee', 'assistant', 'conservator', 'senior'
            ))
    );

    CREATE TABLE artifacts (
        artifact_id TEXT PRIMARY KEY,
        context_id TEXT NOT NULL REFERENCES contexts(context_id),
        material_id TEXT NOT NULL REFERENCES material_types(material_id),
        find_type_id INTEGER NOT NULL REFERENCES find_types(find_type_id),
        catalogue_code TEXT NOT NULL UNIQUE,
        discovered_on TEXT NOT NULL
            CHECK (date(discovered_on) IS NOT NULL),
        estimated_period TEXT NOT NULL
            CHECK (estimated_period IN (
                'prehistoric', 'Roman', 'medieval',
                'post-medieval', 'modern', 'uncertain'
            )),
        condition_grade INTEGER NOT NULL
            CHECK (condition_grade BETWEEN 1 AND 5),
        weight_g REAL NOT NULL CHECK (weight_g > 0),
        length_mm REAL NOT NULL CHECK (length_mm > 0),
        width_mm REAL NOT NULL CHECK (width_mm > 0),
        is_diagnostic INTEGER NOT NULL
            CHECK (is_diagnostic IN (0, 1)),
        storage_status TEXT NOT NULL
            CHECK (storage_status IN (
                'archived', 'on_display', 'awaiting_assessment'
            ))
    );

    CREATE TABLE artifact_assessments (
        assessment_id TEXT PRIMARY KEY,
        artifact_id TEXT NOT NULL REFERENCES artifacts(artifact_id),
        conservator_id TEXT NOT NULL REFERENCES conservators(conservator_id),
        assessed_on TEXT NOT NULL
            CHECK (date(assessed_on) IS NOT NULL),
        condition_grade INTEGER NOT NULL
            CHECK (condition_grade BETWEEN 1 AND 5),
        priority TEXT NOT NULL
            CHECK (priority IN ('routine', 'soon', 'urgent')),
        assessment_note TEXT
    );

    CREATE TABLE conservation_treatments (
        treatment_id TEXT PRIMARY KEY,
        artifact_id TEXT NOT NULL REFERENCES artifacts(artifact_id),
        conservator_id TEXT NOT NULL REFERENCES conservators(conservator_id),
        started_on TEXT NOT NULL
            CHECK (date(started_on) IS NOT NULL),
        completed_on TEXT
            CHECK (
                completed_on IS NULL
                OR date(completed_on) >= date(started_on)
            ),
        treatment_method TEXT NOT NULL
            CHECK (treatment_method IN (
                'dry cleaning', 'stabilisation', 'desalination',
                'adhesive repair', 'protective coating'
            )),
        outcome TEXT NOT NULL
            CHECK (outcome IN (
                'completed', 'in progress', 'monitoring'
            ))
    );

    CREATE TABLE treatment_supplies (
        treatment_id TEXT NOT NULL
            REFERENCES conservation_treatments(treatment_id),
        supply_id TEXT NOT NULL
            REFERENCES conservation_supplies(supply_id),
        quantity_g REAL NOT NULL CHECK (quantity_g > 0),
        PRIMARY KEY (treatment_id, supply_id)
    );

    CREATE TABLE environmental_samples (
        sample_id TEXT PRIMARY KEY,
        context_id TEXT NOT NULL REFERENCES contexts(context_id),
        collected_on TEXT NOT NULL
            CHECK (date(collected_on) IS NOT NULL),
        soil_temperature_c REAL NOT NULL
            CHECK (soil_temperature_c BETWEEN -10 AND 45),
        soil_ph REAL NOT NULL
            CHECK (soil_ph BETWEEN 0 AND 14),
        moisture_percent REAL NOT NULL
            CHECK (moisture_percent BETWEEN 0 AND 100)
    );

    CREATE TABLE context_find_catalogue (
        context_id TEXT NOT NULL REFERENCES contexts(context_id),
        find_type_id INTEGER NOT NULL REFERENCES find_types(find_type_id),
        recovered_count INTEGER NOT NULL CHECK (recovered_count >= 0),
        PRIMARY KEY (context_id, find_type_id)
    );

    CREATE TABLE intake_submissions (
        submission_id TEXT PRIMARY KEY,
        artifact_id TEXT NOT NULL REFERENCES artifacts(artifact_id),
        submitted_at TEXT NOT NULL
            CHECK (date(submitted_at) IS NOT NULL),
        source_reference TEXT NOT NULL,
        source_note TEXT,
        duplicate_of_submission_id TEXT
            REFERENCES intake_submissions(submission_id),
        CHECK (
            duplicate_of_submission_id IS NULL
            OR duplicate_of_submission_id <> submission_id
        )
    );

    CREATE INDEX idx_artifacts_context
        ON artifacts(context_id);
    CREATE INDEX idx_artifacts_material
        ON artifacts(material_id);
    CREATE INDEX idx_assessments_artifact_date
        ON artifact_assessments(artifact_id, assessed_on);
    CREATE INDEX idx_samples_context_date
        ON environmental_samples(context_id, collected_on);
    CREATE INDEX idx_intake_artifact
        ON intake_submissions(artifact_id);
    """)


def seed_reference_data(
    db: sqlite3.Connection,
    rng: random.Random,
) -> tuple[list[str], list[str], list[int]]:
    """Add the fixed reference records used by the generated data."""
    db.executemany(
        "INSERT INTO excavation_sites VALUES (?, ?, ?, ?)",
        SITES,
    )

    trench_rows = []
    for site_id, *_ in SITES:
        for number in range(1, 4):
            trench_rows.append((
                f"{site_id}-T{number:02d}",
                site_id,
                f"Trench {number}",
                round(rng.uniform(12, 68), 1),
            ))

    db.executemany(
        "INSERT INTO trenches VALUES (?, ?, ?, ?)",
        trench_rows,
    )

    periods = [
        "prehistoric", "Roman", "medieval", "post-medieval", "modern"
    ]
    context_rows = []

    for trench_id, *_ in trench_rows:
        for number in range(1, 7):
            context_rows.append((
                f"{trench_id}-C{number:02d}",
                trench_id,
                number,
                rng.choice([
                    "occupation layer",
                    "pit fill",
                    "floor deposit",
                    "ditch fill",
                    "demolition deposit",
                ]),
                rng.choices(periods, weights=[1, 2, 3, 2, 1])[0],
            ))

    db.executemany(
        "INSERT INTO contexts VALUES (?, ?, ?, ?, ?)",
        context_rows,
    )
    db.executemany(
        "INSERT INTO material_types VALUES (?, ?)",
        MATERIALS,
    )
    db.executemany(
        "INSERT INTO conservation_supplies VALUES (?, ?, ?)",
        CONSERVATION_SUPPLIES,
    )
    db.executemany(
        "INSERT INTO find_types(find_type_name) VALUES (?)",
        [(name,) for name in FIND_TYPES],
    )

    type_ids = {
        name: identifier
        for identifier, name in db.execute(
            "SELECT find_type_id, find_type_name FROM find_types"
        )
    }

    conservator_rows = [
        (
            f"CON-{number:03d}",
            rng.choices(
                ["trainee", "assistant", "conservator", "senior"],
                [2, 3, 4, 1],
            )[0],
        )
        for number in range(1, 19)
    ]
    db.executemany(
        "INSERT INTO conservators VALUES (?, ?)",
        conservator_rows,
    )

    context_ids = [row[0] for row in context_rows]
    conservator_ids = [row[0] for row in conservator_rows]
    return context_ids, conservator_ids, list(type_ids.values())


def generate_records(
    db: sqlite3.Connection,
    rng: random.Random,
    context_ids: list[str],
    conservator_ids: list[str],
    type_ids: list[int],
) -> None:
    """Generate artefacts, assessments, samples, treatments, and intake records."""
    artifact_rows = []
    assessment_rows = []
    sample_rows = []
    intake_rows = []
    treatments = []
    context_catalogue = {}

    first_field_day = date(2024, 3, 1)
    missing_note_ids = set(
        rng.sample(range(1, ARTIFACT_COUNT + 1), MISSING_SOURCE_NOTES)
    )
    missing_assessment_note_ids = set(
        rng.sample(range(1, ARTIFACT_COUNT + 1), MISSING_ASSESSMENT_NOTES)
    )

    for number in range(1, ARTIFACT_COUNT + 1):
        artifact_id = f"ART-{number:05d}"
        context_id = rng.choice(context_ids)
        material_id, _ = rng.choices(
            MATERIALS,
            weights=[30, 10, 18, 12, 6, 8, 4, 3, 2, 7],
        )[0]
        find_type_id = rng.choice(type_ids)
        discovered_on = first_field_day + timedelta(
            days=rng.randrange(0, 730)
        )

        estimated_period = rng.choices(
            [
                "prehistoric", "Roman", "medieval",
                "post-medieval", "modern", "uncertain",
            ],
            [1, 2, 3, 2, 1, 1],
        )[0]

        condition_grade = rng.choices(
            [1, 2, 3, 4, 5],
            CONDITION_WEIGHTS,
        )[0]

        weight_g = round(rng.lognormvariate(3.4, 0.85), 2)
        length_mm = round(rng.uniform(8, 240), 1)
        width_mm = round(
            rng.uniform(5, max(6, min(length_mm, 150))),
            1,
        )
        is_diagnostic = 1 if rng.random() < 0.27 else 0

        # Storage status is separate from historical treatment records.
        storage_status = rng.choices(
            ["archived", "on_display", "awaiting_assessment"],
            [0.78, 0.08, 0.14],
        )[0]

        artifact_rows.append((
            artifact_id,
            context_id,
            material_id,
            find_type_id,
            f"CAT-{number:05d}",
            discovered_on.isoformat(),
            estimated_period,
            condition_grade,
            weight_g,
            length_mm,
            width_mm,
            is_diagnostic,
            storage_status,
        ))

        catalogue_key = (context_id, find_type_id)
        context_catalogue[catalogue_key] = (
            context_catalogue.get(catalogue_key, 0) + 1
        )

        conservator_id = rng.choice(conservator_ids)
        assessment_day = discovered_on + timedelta(
            days=rng.randint(1, 30)
        )

        if condition_grade == 1:
            priority = "urgent"
        elif condition_grade == 2:
            priority = "soon"
        else:
            priority = "routine"

        assessment_note = (
            None if number in missing_assessment_note_ids
            else rng.choice([
                "surface stable",
                "needs controlled storage",
                "monitor for corrosion",
                "fragmented edge",
            ])
        )

        assessment_rows.append((
            f"ASM-{number:05d}",
            artifact_id,
            conservator_id,
            assessment_day.isoformat(),
            condition_grade,
            priority,
            assessment_note,
        ))

        # A subset of artefacts has a related environmental sample.
        if number <= 360:
            sample_day = discovered_on + timedelta(
                days=rng.randint(0, 3)
            )
            temperature = round(rng.uniform(5.0, 19.5), 1)
            ph = round(rng.uniform(4.2, 8.6), 2)
            moisture = round(rng.uniform(3, 62), 1)

            sample_rows.append((
                f"SMP-{number:05d}",
                context_id,
                sample_day.isoformat(),
                temperature,
                ph,
                moisture,
            ))

        submission_id = f"SUB-{number:05d}"
        source_reference = f"FIELD-{discovered_on.year}-{number:05d}"

        source_note = (
            None if number in missing_note_ids
            else rng.choice([
                "field register checked",
                "bag label transcribed",
                "catalogue entry verified",
                "context sheet cross-checked",
            ])
        )

        intake_rows.append((
            submission_id,
            artifact_id,
            discovered_on.isoformat(),
            source_reference,
            source_note,
            None,
        ))

        # Retain repeat imports and link them to their original submissions.
        if number <= DUPLICATE_SUBMISSIONS:
            duplicate_id = f"SUB-DUP-{number:03d}"
            intake_rows.append((
                duplicate_id,
                artifact_id,
                (discovered_on + timedelta(days=1)).isoformat(),
                source_reference,
                "repeat import retained for audit",
                submission_id,
            ))

        # Poorer-condition finds are more likely to receive treatment.
        if condition_grade <= 2 and number % 4 == 0:
            treatment_start = assessment_day + timedelta(
                days=rng.randint(1, 12)
            )
            treatment_done = (
                treatment_start + timedelta(days=rng.randint(1, 9))
                if rng.random() < 0.82
                else None
            )

            treatment_method = rng.choice([
                "dry cleaning",
                "stabilisation",
                "desalination",
                "adhesive repair",
                "protective coating",
            ])

            if treatment_done:
                outcome = "completed"
            else:
                outcome = rng.choice(["in progress", "monitoring"])

            treatments.append((
                f"TRT-{len(treatments) + 1:04d}",
                artifact_id,
                conservator_id,
                treatment_start.isoformat(),
                treatment_done.isoformat() if treatment_done else None,
                treatment_method,
                outcome,
            ))

    db.executemany(
        "INSERT INTO artifacts VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        artifact_rows,
    )
    db.executemany(
        "INSERT INTO artifact_assessments VALUES (?, ?, ?, ?, ?, ?, ?)",
        assessment_rows,
    )
    db.executemany(
        "INSERT INTO environmental_samples VALUES (?, ?, ?, ?, ?, ?)",
        sample_rows,
    )
    db.executemany(
        "INSERT INTO intake_submissions VALUES (?, ?, ?, ?, ?, ?)",
        intake_rows,
    )

    supply_ids = [
        row[0]
        for row in db.execute(
            "SELECT supply_id FROM conservation_supplies"
        )
    ]

    for (
        treatment_id,
        artifact_id,
        conservator_id,
        treatment_start,
        treatment_done,
        treatment_method,
        outcome,
    ) in treatments:
        db.execute(
            "INSERT INTO conservation_treatments VALUES (?, ?, ?, ?, ?, ?, ?)",
            (
                treatment_id,
                artifact_id,
                conservator_id,
                treatment_start,
                treatment_done,
                treatment_method,
                outcome,
            ),
        )

        # A treatment may consume several supplies. The composite key
        # prevents the same supply being repeated for that treatment.
        selected_supplies = rng.sample(supply_ids, rng.randint(1, 3))
        for supply_id in selected_supplies:
            db.execute(
                "INSERT INTO treatment_supplies VALUES (?, ?, ?)",
                (
                    treatment_id,
                    supply_id,
                    round(rng.uniform(0.1, 8.0), 2),
                ),
            )

    catalogue_rows = [
        (context_id, find_type_id, count)
        for (context_id, find_type_id), count in sorted(
            context_catalogue.items()
        )
    ]
    db.executemany(
        "INSERT INTO context_find_catalogue VALUES (?, ?, ?)",
        catalogue_rows,
    )


def validate_and_report(db: sqlite3.Connection) -> None:
    """Check database integrity and print counts and data-quality totals."""
    integrity = db.execute("PRAGMA integrity_check").fetchone()[0]
    foreign_key_errors = db.execute(
        "PRAGMA foreign_key_check"
    ).fetchall()

    if integrity != "ok" or foreign_key_errors:
        raise RuntimeError(
            "Database validation failed: "
            f"integrity={integrity}; "
            f"foreign_key_errors={foreign_key_errors[:5]}"
        )

    print("DATABASE VALIDATION")
    print(f"Integrity check: {integrity}")
    print(f"Foreign-key violations: {len(foreign_key_errors)}")

    print("\nTABLE COUNTS")
    table_names = db.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
        ORDER BY name
    """).fetchall()

    for (table_name,) in table_names:
        count = db.execute(
            f'SELECT COUNT(*) FROM "{table_name}"'
        ).fetchone()[0]
        print(f"{table_name}: {count:,}")

    duplicate_count = db.execute("""
        SELECT COUNT(*)
        FROM intake_submissions
        WHERE duplicate_of_submission_id IS NOT NULL
    """).fetchone()[0]

    missing_source_note_count = db.execute("""
        SELECT COUNT(*)
        FROM intake_submissions
        WHERE duplicate_of_submission_id IS NULL
          AND source_note IS NULL
    """).fetchone()[0]

    original_count = db.execute("""
        SELECT COUNT(*)
        FROM intake_submissions
        WHERE duplicate_of_submission_id IS NULL
    """).fetchone()[0]

    total_submissions = original_count + duplicate_count

    missing_assessment_note_count = db.execute("""
        SELECT COUNT(*)
        FROM artifact_assessments
        WHERE assessment_note IS NULL
    """).fetchone()[0]

    assessment_count = db.execute(
        "SELECT COUNT(*) FROM artifact_assessments"
    ).fetchone()[0]

    print("\nDATA QUALITY RECORDS")
    print(
        f"Duplicate submissions: {duplicate_count:,} "
        f"({duplicate_count / total_submissions:.1%} of all submissions)"
    )
    print(
        f"Original submissions missing an optional source note: "
        f"{missing_source_note_count:,} "
        f"({missing_source_note_count / original_count:.1%} of original submissions)"
    )
    print(
        f"Assessments missing an optional note: "
        f"{missing_assessment_note_count:,} "
        f"({missing_assessment_note_count / assessment_count:.1%} of assessments)"
    )

    print("\nARTIFACT TABLE COLUMNS")
    for column in db.execute("PRAGMA table_info(artifacts)"):
        primary_key = " PRIMARY KEY" if column[5] else ""
        not_null = " NOT NULL" if column[3] else ""
        print(f"{column[1]} {column[2]}{primary_key}{not_null}")


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate the synthetic UK archaeological finds database."
    )
    parser.add_argument(
        "--output",
        type=Path,
        help="SQLite output path; defaults to the current folder.",
    )

    # Jupyter and Colab add their own kernel arguments, such as -f.
    # Ignore those while still recognizing the optional --output argument.
    args, _unknown_args = parser.parse_known_args()

    # The database is saved in the current folder unless --output is supplied.
    output_path = args.output or Path.cwd() / "uk_archaeological_finds.sqlite"

    rng = random.Random(SEED)
    db = connect_database(output_path)

    try:
        create_schema(db)
        context_ids, conservator_ids, type_ids = seed_reference_data(db, rng)
        generate_records(
            db,
            rng,
            context_ids,
            conservator_ids,
            type_ids,
        )
        db.commit()
        validate_and_report(db)
    except Exception:
        db.rollback()
        raise
    finally:
        db.close()

    print(f"\nDatabase saved to: {output_path.resolve()}")


if __name__ == "__main__":
    main()

DATABASE VALIDATION
Integrity check: ok
Foreign-key violations: 0

TABLE COUNTS
artifact_assessments: 1,800
artifacts: 1,800
conservation_supplies: 6
conservation_treatments: 97
conservators: 18
context_find_catalogue: 613
contexts: 108
environmental_samples: 360
excavation_sites: 6
find_types: 6
intake_submissions: 1,836
material_types: 10
treatment_supplies: 199
trenches: 18

DATA QUALITY RECORDS
Duplicate submissions: 36 (2.0% of all submissions)
Original submissions missing an optional source note: 144 (8.0% of original submissions)
Assessments missing an optional note: 360 (20.0% of assessments)

ARTIFACT TABLE COLUMNS
artifact_id TEXT PRIMARY KEY
context_id TEXT NOT NULL
material_id TEXT NOT NULL
find_type_id INTEGER NOT NULL
catalogue_code TEXT NOT NULL
discovered_on TEXT NOT NULL
estimated_period TEXT NOT NULL
condition_grade INTEGER NOT NULL
weight_g REAL NOT NULL
length_mm REAL NOT NULL
width_mm REAL NOT NULL
is_diagnostic INTEGER NOT NULL
storage_status TEXT NOT NULL

Databa